# Pull Request Title and Body

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import math

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)

sns.set_theme(style='whitegrid', context='notebook', palette='husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

C:\Users\sydne\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\seaborn\_statistics.py:32: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.5.4)
  from scipy.stats import gaussian_kde


#### Load Dataset

This section loads the public and private pull requests. We look at data from PRs collected from **April 2026**. We also make sure PRs are unique. The below code is accessed from the database.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data")

csv_files = sorted(DATA_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No code complexity CSV files found in {DATA_DIR}")

df = pd.concat(
    (pd.read_csv(path) for path in csv_files),
    ignore_index=True
)

print(f"Loaded {len(csv_files):,} CSV files")
print(f"Rows: {len(df):,}")

df.head(20)

In [1]:
import pandas as pd

# Load the cleaned dataset
df = pd.read_csv("../data/cleaned_public.csv")

# Preview the first five rows
df.head()

,repository_id,pull_request_id,is_open_source,title,body,merge_day,merge_month,files_added,files_deleted,files_modified,...,body_lower,detected_language,title_clean_nlp,body_clean_nlp,combined_text_nlp,title_word_count,body_word_count,has_title,has_body,short_body
0,455390871,3442120599,True,Limit tunnel IDs and names to 49 characters fo...,DNS specifications limit each label in a hostn...,2026-03-26,2026-03-01,0,0,10,...,dns specifications limit each label in a hostn...,en,limit tunnel names characters dns compliance,dns specifications limit each label hostname c...,limit tunnel names characters dns compliance d...,11,160,True,True,False
1,36487440,3415836335,True,[master] fix: time period set to none by default,This is an automatic copy of pull request #157...,2026-03-18,2026-03-01,0,0,1,...,this is an automatic copy of pull request #157...,en,master fix time period default,automatic copy done mergify issue description ...,master fix time period default automatic copy ...,9,55,True,True,False
2,892864551,3392350402,True,docs: Add tail sampling example,#### Description\r\nAdded the `examples/tail-s...,2026-03-16,2026-03-01,4,0,4,...,#### description\r\nadded the `examples/tail-s...,en,docs tail sampling example,description directory contain material referen...,docs tail sampling example description directo...,5,75,True,True,False
3,1076973608,3373905540,True,Bump go.mod version to n-1,"Also bump CI, and dropped useless `name:`s and...",2026-03-09,2026-03-01,0,0,2,...,"also bump ci, and dropped useless `name:`s and...",en,bump go.mod version,bump dropped useless default tempting swap ver...,bump go.mod version bump dropped useless defau...,7,35,True,True,False
4,1174469038,3368258841,True,Add ripple effect animation to prompt area chips,## Summary\nEnhanced the visual feedback for c...,2026-03-07,2026-03-01,0,0,4,...,## summary\nenhanced the visual feedback for c...,en,ripple effect animation prompt area chips,summary enhanced visual feedback chip interact...,ripple effect animation prompt area chips summ...,8,211,True,True,False


### Dataset Features
Display the total number of features and list all column names in the dataset.

In [2]:
# Number of features
print(f"Number of features: {df.shape[1]}")

# List each feature
for column in df.columns:
    print(column)

Number of features: 37
repository_id
pull_request_id
is_open_source
title
body
merge_day
merge_month
files_added
files_deleted
files_modified
files_touched
lines_modified
code_lines_modified
config_lines_modified
minutes_to_merge
num_reviewers
num_review_comments
is_code_pr
complexity_score
lines_added_by_language
lines_deleted_by_language
languages_list
had_copilot_seat
user_id
title_clean
body_clean
title_lower
body_lower
detected_language
title_clean_nlp
body_clean_nlp
combined_text_nlp
title_word_count
body_word_count
has_title
has_body
short_body


## Clean title and body text
In this section we walk through how we can process the data from the titles and body of the data in order to gain some insights. We hope to draw conclusion about the relationships between the lines of code added in the pull request, the time to time merge and the contents of the title/body of the pull requests. 

In [4]:
print(f"Unique repositories: {df['repository_id'].nunique():,}")

Unique repositories: 61,941


#### 2a. Number of repositories

#### 2b. Generate descriptive statistics table

In [5]:
import ast


def numeric_series(column):
    if column not in df.columns:
        return pd.Series(np.nan, index=df.index, dtype="float64")
    return pd.to_numeric(df[column], errors="coerce")


def language_count(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (list, tuple, set)):
        return len(value)
    text = str(value).strip()
    if not text:
        return np.nan
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (list, tuple, set)):
            return len(parsed)
    except (ValueError, SyntaxError):
        pass
    return len([item for item in text.replace(";", ",").split(",") if item.strip()])


language_values = numeric_series("languages")
if language_values.notna().sum() == 0 and "languages_list" in df.columns:
    language_values = df["languages_list"].apply(language_count)

semantic_column = next(
    (
        column
        for column in [
            "semantic_breadth",
            "keyword_category_count",
            "complexity_category_breadth",
        ]
        if column in df.columns
    ),
    None,
)

metrics = {
    "Code lines modified": numeric_series("code_lines_modified"),
    "Configuration lines modified": numeric_series("config_lines_modified"),
    "Files touched": numeric_series("files_touched"),
    "Programming languages": language_values,
    "Reviewers": numeric_series("num_reviewers"),
    "Review comments": numeric_series("num_review_comments"),
    "Merge time (minutes)": numeric_series("minutes_to_merge"),
}

summary_rows = []
for variable, values in metrics.items():
    values = values.dropna()
    summary_rows.append(
        {
            "Variable": variable,
            "Mean": values.mean() if not values.empty else np.nan,
            "Median": values.median() if not values.empty else np.nan,
            "SD": values.std() if not values.empty else np.nan,
            "IQR": values.quantile(0.75) - values.quantile(0.25) if not values.empty else np.nan,
        }
    )

summary = pd.DataFrame(summary_rows)


def format_latex_value(value):
    return "--" if pd.isna(value) else f"{value:.2f}"


latex_rows = [
    "\\begin{table}[t]",
    "\\centering",
    "\\caption{Descriptive statistics for the pull request dataset.}",
    "\\label{tab:dataset_summary}",
    "\\small",
    "\\begin{tabular}{lrrrr}",
    "\\toprule",
    "\\textbf{Variable} & \\textbf{Mean} & \\textbf{Median} & \\textbf{SD} & \\textbf{IQR} \\\\",
    "\\midrule",
]

for row in summary.itertuples(index=False):
    latex_rows.append(
        f"{row.Variable} & {format_latex_value(row.Mean)} & "
        f"{format_latex_value(row.Median)} & {format_latex_value(row.SD)} & "
        f"{format_latex_value(row.IQR)} \\\\\\" 
    )

latex_rows.extend([
    "\\bottomrule",
    "\\end{tabular}",
    "\\end{table}",
])

latex_table = "\n".join(latex_rows)
print(latex_table)
summary

\begin{table}[t]
\centering
\caption{Descriptive statistics for the pull request dataset.}
\label{tab:dataset_summary}
\small
\begin{tabular}{lrrrr}
\toprule
\textbf{Variable} & \textbf{Mean} & \textbf{Median} & \textbf{SD} & \textbf{IQR} \\
\midrule
Code lines modified & 304.59 & 6.00 & 1249.30 & 128.00 \\\
Configuration lines modified & 128.29 & 5.00 & 662.07 & 42.00 \\\
Files touched & 11.09 & 3.00 & 38.14 & 6.00 \\\
Programming languages & 1.80 & 1.00 & 1.42 & 1.00 \\\
Reviewers & 1.07 & 1.00 & 1.05 & 2.00 \\\
Review comments & 1.30 & 0.00 & 5.72 & 0.00 \\\
Merge time (minutes) & 1859.17 & 135.00 & 4276.56 & 1349.00 \\\
\bottomrule
\end{tabular}
\end{table}


,Variable,Mean,Median,SD,IQR
0,Code lines modified,304.592479,6.0,1249.303147,128.0
1,Configuration lines modified,128.286768,5.0,662.072423,42.0
2,Files touched,11.085715,3.0,38.136058,6.0
3,Programming languages,1.803226,1.0,1.421334,1.0
4,Reviewers,1.071262,1.0,1.049824,2.0
5,Review comments,1.301336,0.0,5.718483,0.0
6,Merge time (minutes),1859.174103,135.0,4276.555771,1349.0


#### Checking for Missing Values, Non-English & Duplicates
This section compares missing values across the dataset and checks that each PR is unique. In this workflow, I pull data from multiple SQL queries. Since I am using multiple CSV files, there is a chance that the same PR may be pulled in more than once. To prevent this, I check for duplicate PRs across all CSV files. One problem is I also pull non-English PRs.

In [6]:
import re
import numpy as np
import pandas as pd

from langdetect import DetectorFactory, LangDetectException, detect
from tqdm import tqdm

# Enable tqdm progress bars for pandas
tqdm.pandas()

DetectorFactory.seed = 42

TITLE_COL = "title"
BODY_COL = "body"


# ============================================================
# SET UP DATAFRAME
# ============================================================

missing_text_cols = [
    col for col in [TITLE_COL, BODY_COL]
    if col not in df.columns
]

if missing_text_cols:
    raise ValueError(f"Missing text columns: {missing_text_cols}")


# ============================================================
# STOPWORDS
# ============================================================

DEFAULT_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "has", "have",
    "in", "into", "is", "it", "its", "of", "on", "or", "that", "the", "this", "to",
    "was", "were", "will", "with", "without", "we", "you", "your", "our", "their",
    "pr", "pull", "request", "merge", "merged", "branch", "commit", "commits", "github",
    "com", "http", "https", "www", "url", "urls", "href", "hrefs", "id", "ids", "file",
    "files", "code", "change", "changes", "update", "updates", "updated", "add", "added",
    "remove", "removed", "use", "using", "used", "new", "old", "make", "made", "set",
    "get", "also", "one", "two", "true", "false", "none", "null", "na", "n", "amp",
    "nbsp", "src", "lib", "app", "main", "test", "tests", "spec", "specs", "tmp", "var",
    "const", "let", "function", "class", "method", "api", "utils", "util", "service",
    "component", "module", "object", "data", "value", "values", "type", "types", "bot",
    "ref", "refs", "issue_ref", "ticket_ref",
}

EXTRA_STOPWORDS = set()

STOPWORDS = DEFAULT_STOPWORDS | EXTRA_STOPWORDS

NOISE_TOKEN_PARTS = ("bot",)


# ============================================================
# TEXT CLEANING FUNCTIONS
# ============================================================

def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    text = re.sub(
        r"```.*?```",
        " ",
        text,
        flags=re.DOTALL
    )

    text = re.sub(
        r"`[^`]+`",
        " ",
        text
    )

    text = re.sub(
        r"&nbsp;|\bnbsp\b",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"https?://\S+",
        " URL ",
        text
    )

    text = re.sub(
        r"#[0-9]+",
        " ",
        text
    )

    text = re.sub(
        r"\b[A-Z][A-Z0-9]+-\d+\b",
        " ",
        text
    )

    text = (
        text
        .replace("_", " ")
        .replace("-", " ")
        .replace("/", " ")
    )

    text = re.sub(
        r"[^A-Za-z0-9#+. ]+",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip().lower()


def tokenize(text, min_len=3):

    raw_terms = re.findall(
        r"[a-z][a-z0-9+#.]*",
        str(text).lower()
    )

    terms = [
        term.strip("#+.")
        for term in raw_terms
    ]

    return [
        term
        for term in terms
        if len(term) >= min_len
        and term not in STOPWORDS
        and not term.isdigit()
        and not any(
            noise_part in term
            for noise_part in NOISE_TOKEN_PARTS
        )
    ]


def clean_for_nlp(text):
    return " ".join(
        tokenize(
            normalize_text(text)
        )
    )


def detect_pr_language(row):

    text = f"{row[TITLE_COL]} {row[BODY_COL]}".strip()

    text = normalize_text(text)

    if len(text) < 20:
        return "unknown"

    try:
        return detect(text)

    except LangDetectException:
        return "unknown"


def word_count(text):

    text = "" if pd.isna(text) else str(text)

    return len(
        re.findall(
            r"\b\w+\b",
            text
        )
    )


def q90(values):

    values = pd.to_numeric(
        values,
        errors="coerce"
    ).dropna()

    return (
        values.quantile(0.90)
        if len(values)
        else np.nan
    )


# ============================================================
# NUMERIC COLUMNS
# ============================================================

numeric_cols = [
    "minutes_to_merge",
    "lines_added",
    "lines_deleted",
    "files_added",
    "files_deleted",
    "files_modified",
    "num_reviewers",
    "num_review_comments"
]

for col in numeric_cols:

    if col in df.columns:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )


# ============================================================
# PR SIZE / COMPLEXITY VARIABLES
# ============================================================

zero_series = pd.Series(
    0,
    index=df.index
)

lines_added = (
    df["lines_added"]
    if "lines_added" in df.columns
    else zero_series
)

lines_deleted = (
    df["lines_deleted"]
    if "lines_deleted" in df.columns
    else zero_series
)

files_added = (
    df["files_added"]
    if "files_added" in df.columns
    else zero_series
)

files_deleted = (
    df["files_deleted"]
    if "files_deleted" in df.columns
    else zero_series
)

files_modified = (
    df["files_modified"]
    if "files_modified" in df.columns
    else zero_series
)


if "lines_modified" not in df.columns:

    df["lines_modified"] = (
        lines_added.fillna(0)
        +
        lines_deleted.fillna(0)
    )


if "files_touched" not in df.columns:

    df["files_touched"] = (
        files_added.fillna(0)
        +
        files_deleted.fillna(0)
        +
        files_modified.fillna(0)
    )


if "complexity_score" not in df.columns:

    df["complexity_score"] = (
        df["lines_modified"]
        +
        (10 * df["files_touched"])
    )


# ============================================================
# REQUIRED COLUMNS
# ============================================================

if "user_id" not in df.columns:

    if "author_id" in df.columns:
        df["user_id"] = df["author_id"]

    else:
        df["user_id"] = np.arange(len(df))


for col in [
    "pull_request_id",
    "repository_id",
    "num_reviewers",
    "num_review_comments",
    "minutes_to_merge"
]:

    if col not in df.columns:
        df[col] = np.nan


# ============================================================
# BASIC TEXT CLEANING
# ============================================================

df[TITLE_COL] = (
    df[TITLE_COL]
    .fillna("")
    .astype(str)
)

df[BODY_COL] = (
    df[BODY_COL]
    .fillna("")
    .astype(str)
)


df["title_clean"] = (
    df[TITLE_COL]
    .str.strip()
)

df["body_clean"] = (
    df[BODY_COL]
    .str.strip()
)

df["title_lower"] = (
    df["title_clean"]
    .str.lower()
)

df["body_lower"] = (
    df["body_clean"]
    .str.lower()
)


# ============================================================
# LANGUAGE DETECTION
# ============================================================

print("Detecting PR languages...")

df["detected_language"] = df.progress_apply(
    detect_pr_language,
    axis=1
)


language_counts_before_filter = (
    df["detected_language"]
    .value_counts(dropna=False)
)


rows_before_language_filter = len(df)


df = (
    df[
        df["detected_language"].eq("en")
    ]
    .reset_index(drop=True)
)


rows_removed_non_english = (
    rows_before_language_filter
    -
    len(df)
)


# ============================================================
# NLP CLEANING
# ============================================================

print("\nCleaning PR titles...")

df["title_clean_nlp"] = (
    df[TITLE_COL]
    .progress_apply(clean_for_nlp)
)


print("\nCleaning PR bodies...")

df["body_clean_nlp"] = (
    df[BODY_COL]
    .progress_apply(clean_for_nlp)
)


df["combined_text_nlp"] = (
    df["title_clean_nlp"]
    +
    " "
    +
    df["body_clean_nlp"]
).str.strip()


# ============================================================
# WORD COUNTS
# ============================================================

print("\nCounting title words...")

df["title_word_count"] = (
    df["title_clean"]
    .progress_apply(word_count)
)


print("\nCounting body words...")

df["body_word_count"] = (
    df["body_clean"]
    .progress_apply(word_count)
)


# ============================================================
# TEXT FLAGS
# ============================================================

df["has_title"] = (
    df["title_word_count"] > 0
)

df["has_body"] = (
    df["body_word_count"] > 0
)

df["short_body"] = (
    df["body_word_count"] < 20
)


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 60)

print(
    f"Rows before language filter: "
    f"{rows_before_language_filter:,}"
)

print(
    f"Rows after English-only filter: "
    f"{len(df):,}"
)

print(
    f"Rows removed as non-English or unknown: "
    f"{rows_removed_non_english:,}"
)

print("=" * 60)


display(
    language_counts_before_filter
    .head(20)
    .rename_axis("detected_language")
    .reset_index(name="pr_count")
)


display(
    df[
        [
            TITLE_COL,
            BODY_COL,
            "detected_language",
            "title_word_count",
            "body_word_count",
            "combined_text_nlp"
        ]
    ].head()
)

Detecting PR languages...


100%|██████████| 1400000/1400000 [2:11:00<00:00, 178.10it/s] 



Cleaning PR titles...


100%|██████████| 1338401/1338401 [00:44<00:00, 30077.08it/s]



Cleaning PR bodies...


...


Counting body words...


100%|██████████| 1338401/1338401 [09:00<00:00, 2476.10it/s]



Rows before language filter: 1,400,000
Rows after English-only filter: 1,338,401
Rows removed as non-English or unknown: 61,599


,detected_language,pr_count
0,en,1338401
1,unknown,18860
2,fr,7834
3,no,7698
4,it,4324
5,ca,4247
6,ro,3505
7,da,3262
8,sv,2063
9,es,1818


,title,body,detected_language,title_word_count,body_word_count,combined_text_nlp
0,Limit tunnel IDs and names to 49 characters for DNS compliance,DNS specifications limit each label in a hostname to 63 characters. Tunnel URIs include the tunn...,en,11,160,limit tunnel names characters dns compliance dns specifications limit each label hostname charac...
1,[master] fix: time period set to none by default,This is an automatic copy of pull request #15771 done by [Mergify](https://mergify.com).\n\n----...,en,9,55,master fix time period default automatic copy done mergify issue description noticed runtime log...
2,docs: Add tail sampling example,#### Description\r\nAdded the `examples/tail-sampling` directory that will contain material to b...,en,5,75,docs tail sampling example description directory contain material referenced customer documentat...
3,Bump go.mod version to n-1,"Also bump CI, and dropped useless `name:`s and default `fetch-depth`.\r\n\r\nTempting to swap th...",en,7,35,bump go.mod version bump dropped useless default tempting swap versions setup aliases but going ...
4,Add ripple effect animation to prompt area chips,## Summary\nEnhanced the visual feedback for chip interactions in the prompt area by adding a ri...,en,8,211,ripple effect animation prompt area chips summary enhanced visual feedback chip interactions pro...


## Clean title and body text
In this section we walk through how we can process the data from the titles and body of the data in order to gain some insights. We hope to draw conclusion about the relationships between the lines of code added in the pull request, the time to time merge and the contents of the title/body of the pull requests. 

In [7]:
output_file = "../data/cleaned_public.csv"

df.to_csv(
    output_file,
    index=False
)

print(f"Saved {len(df):,} PRs to {output_file}")

Saved 1,338,401 PRs to ../data/cleaned_public.csv
